(sec_ex_30_1)=

# Exercise 30.1: The basic Razumova model

In this exercise, we will implement the simple four-state crossbridge model from Razumova et al. (2000) — without cooperativity. This will help us understand the fundamental dynamics before exploring the cooperative mechanisms in Exercise 30.2.

Refer to the theory in [The stiffness-distortion crossbridge model](sec_razumova_basic) for the equations.


## 30.1a: Implement and solve the basic model

The simple Razumova model tracks three state variables: $D$ (detached), $A_1$ (pre-powerstroke), and $A_2$ (post-powerstroke). The fourth state, $R_{\mathrm{off}}$, is found from mass conservation.

**Your tasks:**

1. Fill in the initial conditions, rate constants, and the right-hand side of the ODE system.
2. Solve the system using `solve_ivp`.
3. Plot the time evolution of all three state variables.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# Initial conditions (all crossbridges start in the off state)
D_0 = ...  # Fill in
A1_0 = ...  # Fill in
A2_0 = ...  # Fill in

# Rate constants
R_T = 1  # Total regulatory units (normalised)
k_on = 400  # Rate from non-permissive to permissive (1/sec)
k_off = 50  # Rate from permissive to non-permissive (1/sec)
f = 50  # Attachment rate (1/sec)
f_prime = 400  # Reverse attachment rate (1/sec)
h = 8  # Powerstroke rate (1/sec)
h_prime = 6  # Reverse powerstroke rate (1/sec)
g = 4  # Detachment rate (1/sec)

In [ ]:
def rhs(t, y):
    """Right-hand side of the basic Razumova model."""
    D, A_1, A_2 = y

    # Mass conservation: R_off = R_T - D - A_1 - A_2
    R_off = ...

    # ODEs (fill in the right-hand sides using the equations from the theory)
    dD_dt = ...
    dA1_dt = ...
    dA2_dt = ...

    return [dD_dt, dA1_dt, dA2_dt]

In [ ]:
# Solve the ODE system
t_span = (0, 10)
t_eval = np.linspace(*t_span, 5000)
y0 = [D_0, A1_0, A2_0]

sol = solve_ivp(rhs, t_span, y0, t_eval=t_eval, method="RK45")

# Extract the solution components
D = sol.y[0]
A_1 = sol.y[1]
A_2 = sol.y[2]
time = sol.t

In [ ]:
# Plot all three state variables
plt.figure(figsize=(8, 5))
plt.plot(time, D, label=r"$D$")
plt.plot(time, A_1, label=r"$A_1$")
plt.plot(time, A_2, label=r"$A_2$")
plt.xlabel("Time (s)")
plt.ylabel("State probability")
plt.title("Basic Razumova model — State probabilities")
plt.ylim(0, 1)
plt.legend()
plt.show()

## 30.1b: Rate of tension development

In this simplified model, the total force produced by the system is proportional to the number of XBs in the post-powerstroke $A_2$ state.

**Your tasks:**

1. Plot the time course of force development (i.e., $A_2(t)$).
2. Calculate the rate of tension development $k_{\mathrm{dev}}$ by finding the time $t_{63\%}$ at which the force reaches $(1-1/e) \approx 63\%$ of its maximum value, then computing $k_{\mathrm{dev}} = 1/t_{63\%}$.


In [ ]:
# Plot the force development
plt.figure(figsize=(8, 5))
plt.plot(time, A_2, label="Relative force", color="C3")
plt.xlabel("Time (s)")
plt.ylabel("Relative force ($A_2$)")
plt.title("Time course of force development")
plt.xlim(0, 1)
plt.legend()
plt.show()

# Calculate k_dev
f_max = A_2[-1]  # Steady-state maximum force
f_63 = (1 - 1 / np.e) * f_max  # 63% of max

# Find the first index where A_2 exceeds the 63% threshold
idx = np.searchsorted(A_2, f_63)
t_63 = time[idx]
k_dev = 1 / t_63

print(f"Steady-state force (A_2_max): {f_max:.4f}")
print(f"k_dev = {k_dev:.2f} 1/sec")

## 30.1c: Reflection

This simplified model uses constant kinetic rates and is missing components necessary to match a number of experimental data sets — most importantly, the **steady-state force-pCa curve**.

![Force-pCa curve](../../fig/Force_pCa_Raz.png)

<center><i>Experimentally measured steady-state force-pCa curve for cardiac muscle.</i></center>

**Questions:**

1. Could this model (with constant kinetic rates) reproduce the steep, sigmoidal force-pCa curve shown above? What mechanisms do you think would need to be added?
2. Why is the concept of _cooperativity_ important for crossbridge dynamics?


## 30.1d: Interactive exploration

Use the interactive widget below to explore how the kinetic rate constants affect the state probabilities and force development dynamics.

**Observe and reflect:**

- How does changing $k_{\mathrm{on}}$ affect the equilibrium distribution between states?
- What is the effect of changing the attachment rate $f$ on the rate of force development?
- How do $h$ and $g$ affect the balance between $A_1$ and $A_2$?

**Question:**

3. What happens to the steady-state $A_2$ value if you increase $k_{\mathrm{on}}$? What about if you increase $f$?


In [ ]:
import ipywidgets as widgets


def razumova_basic_widget(k_on=400, f=50, h=8, g=4):
    """Solve and plot the basic Razumova model with given parameters."""
    R_T = 1
    k_off = 50
    f_prime = 400
    h_prime = 6

    def rhs_widget(t, y):
        D, A_1, A_2 = y
        R_off = R_T - D - A_1 - A_2
        dD_dt = k_on * R_off + f_prime * A_1 + g * A_2 - (k_off + f) * D
        dA1_dt = f * D + h_prime * A_2 - (f_prime + h) * A_1
        dA2_dt = h * A_1 - (h_prime + g) * A_2
        return [dD_dt, dA1_dt, dA2_dt]

    t_span = (0, 10)
    t_eval = np.linspace(*t_span, 5000)
    sol = solve_ivp(rhs_widget, t_span, [0, 0, 0], t_eval=t_eval, method="RK45")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

    # State probabilities
    axes[0].plot(sol.t, sol.y[0], label=r"$D$")
    axes[0].plot(sol.t, sol.y[1], label=r"$A_1$")
    axes[0].plot(sol.t, sol.y[2], label=r"$A_2$")
    axes[0].set(
        xlabel="Time (s)",
        ylabel="State probability",
        title="State probabilities",
        ylim=(0, 1),
    )
    axes[0].legend()

    # Force development
    A_2_sol = sol.y[2]
    axes[1].plot(sol.t, A_2_sol, color="C3", label="Relative force")
    axes[1].set(
        xlabel="Time (s)",
        ylabel="Relative force ($A_2$)",
        xlim=(0, 1),
        ylim=(0, 1),
    )

    # k_dev calculation
    f_max = A_2_sol[-1]
    if f_max > 0:
        f_63 = (1 - 1 / np.e) * f_max
        idx = np.searchsorted(A_2_sol, f_63)
        if idx < len(sol.t):
            t_63 = sol.t[idx]
            axes[1].axhline(f_63, color="gray", linestyle="--", alpha=0.5)
            axes[1].axvline(t_63, color="gray", linestyle="--", alpha=0.5)
            axes[1].set_title(f"Force development ($k_{{dev}}$ = {1 / t_63:.1f} 1/s)")
        else:
            axes[1].set_title("Force development")
    else:
        axes[1].set_title("Force development")

    plt.show()


widgets.interact(
    razumova_basic_widget,
    k_on=widgets.FloatSlider(value=400, min=100, max=500, step=10, description=r"k_on"),
    f=widgets.FloatSlider(value=50, min=10, max=200, step=5, description="f"),
    h=widgets.FloatSlider(value=8, min=1, max=30, step=1, description="h"),
    g=widgets.FloatSlider(value=4, min=1, max=20, step=1, description="g"),
);